# F08: File-Drop Ingestion Testing (Fabric Scenario)

## What You'll Learn

A common Fabric ingestion pattern is the **file-drop pipeline**: upstream systems
deposit files into a landing zone (OneLake, ADLS, or a Lakehouse `Files/` folder)
on a daily cadence, and a Data Pipeline or Spark notebook picks them up, validates
them, and loads them into tables.

Testing this pattern requires **realistic file drops** — not just data, but the
full operational envelope: correct file names, manifests, done-flags, late arrivals,
and the occasional corrupted file.

In this notebook you will:

1. **Understand** the file-drop ingestion pattern and its testing challenges
2. **Generate** a healthcare dataset as the source data
3. **Configure** a 14-day daily file-drop simulation
4. **Simulate** the drops with late arrivals
5. **Inject** chaos into selected files
6. **Validate** with validation gates
7. **Review** the complete pipeline test structure

## The File-Drop Pattern

```
landing/
  healthcare/
    2024-01-01/
      patient.parquet
      encounter.parquet
      _manifest.json
      _done
    2024-01-02/
      ...
```

## Prerequisites

- `sqllocks-spindle` installed
- Familiarity with Fabric Data Pipelines (helpful but not required)

## Time Estimate

**~15 minutes**

In [1]:
# Uncomment the line below if sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

Installation cell ready. Uncomment the pip line above if needed.


## Step 1 — Generate Healthcare Source Data

**What we're about to do:** Generate a healthcare dataset that will serve as
the source for our file-drop simulation. This represents the data that an
upstream EHR system would produce daily.

**Why this matters:** The file-drop simulator needs a source dataset to
partition into daily slices. By using the healthcare domain, we get realistic
patient, encounter, and claim data — exactly what a hospital's data feed
would contain.

**What to expect:** A standard healthcare dataset at small scale.

In [2]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.domains.healthcare import HealthcareDomain

result = Spindle().generate(domain=HealthcareDomain(), scale="small", seed=42)

print("Healthcare Source Data")
print("=" * 45)
for table_name, df in result.tables.items():
    print(f"  {table_name:<20} {len(df):>8,} rows")

total = sum(len(df) for df in result.tables.values())
print(f"\nTotal rows: {total:,}")
print(f"\nThis data will be sliced into 14 daily file drops.")

Healthcare Source Data
  facility                   50 rows
  patient                   500 rows
  provider                  100 rows
  encounter               2,500 rows
  claim                   2,375 rows
  diagnosis               4,500 rows
  medication              2,250 rows
  procedure               3,000 rows
  claim_line              5,937 rows

Total rows: 21,212

This data will be sliced into 14 daily file drops.


## Step 2 — Configure the File-Drop Simulation

**What we're about to do:** Set up a `FileDropConfig` that defines every aspect
of the simulation — cadence, date range, output format, lateness behavior,
manifest files, and done-flags.

**Why this matters:** Each configuration parameter maps to a real operational
concern:
- **`cadence`**: How often files land (daily, hourly, etc.)
- **`lateness_enabled` / `lateness_probability`**: Simulates upstream delays
  that your pipeline must handle gracefully
- **`manifest_enabled`**: Produces a `_manifest.json` listing expected files
  and row counts for validation
- **`done_flag_enabled`**: Creates a `_done` sentinel file that signals
  "all files for this batch have landed"

**What to expect:** A config object ready to drive the simulation.

In [3]:
from sqllocks_spindle.simulation.file_drop import FileDropSimulator, FileDropConfig

config = FileDropConfig(
    domain="healthcare",
    base_path="./f08_demo",
    cadence="daily",
    date_range_start="2024-01-01",
    date_range_end="2024-01-14",
    formats=["parquet"],
    lateness_enabled=True,
    lateness_probability=0.15,     # 15% of drops will be late
    manifest_enabled=True,          # produce _manifest.json per batch
    done_flag_enabled=True,         # produce _done sentinel file
    seed=42,
)

print("File-Drop Configuration")
print("=" * 45)
print(f"  Domain:        {config.domain}")
print(f"  Base path:     {config.base_path}")
print(f"  Cadence:       {config.cadence}")
print(f"  Date range:    {config.date_range_start} to {config.date_range_end}")
print(f"  Format:        {config.formats}")
print(f"  Late arrivals: {config.lateness_enabled} ({config.lateness_probability:.0%} probability)")
print(f"  Manifest:      {config.manifest_enabled}")
print(f"  Done flag:     {config.done_flag_enabled}")

File-Drop Configuration
  Domain:        healthcare
  Base path:     ./f08_demo
  Cadence:       daily
  Date range:    2024-01-01 to 2024-01-14
  Format:        ['parquet']
  Late arrivals: True (15% probability)
  Manifest:      True
  Done flag:     True


## Step 3 — Run the Simulation

**What we're about to do:** Execute the file-drop simulation. The simulator
will partition the source data into daily batches, write Parquet files to
date-partitioned folders, generate manifests and done-flags, and randomly
delay some drops to simulate late arrivals.

**Why this matters:** This creates the exact folder structure your Fabric
Data Pipeline would encounter in production. You can point your pipeline
at this output and test its behavior end-to-end — including how it handles
the 15% of drops that arrive late.

**What to expect:** A result showing all files written, any late arrivals,
and the folder structure.

In [4]:
sim = FileDropSimulator(result.tables, config)
drop_result = sim.run()

print(f"Simulation Complete")
print(f"=" * 50)
print(f"  Total files written:  {len(drop_result.files_written):,}")
print(f"  Manifests written:    {len(drop_result.manifest_paths)}")
print(f"  Done flags written:   {len(drop_result.done_flag_paths)}")

# Per-entity statistics from drop_result.stats
print(f"\n=== Per-Entity Statistics ===")
print(f"{'Entity':<25} {'Files':>8} {'Rows Written':>14} {'Formats'}")
print("-" * 65)
for entity_name, entity_stats in drop_result.stats.items():
    print(f"  {entity_name:<23} {entity_stats['files']:>8} {entity_stats['rows_written']:>14,}   {entity_stats['formats']}")

total_rows = sum(s['rows_written'] for s in drop_result.stats.values())
print(f"\nTotal entities: {len(drop_result.stats)}")
print(f"Total rows written: {total_rows:,}")
print(f"Total data files: {len(drop_result.files_written)}")

Simulation Complete
  Total files written:  302
  Manifests written:    105
  Done flags written:   105

=== Per-Entity Statistics ===
Entity                       Files   Rows Written Formats
-----------------------------------------------------------------
  facility                      19             44   ['parquet']
  patient                        0              0   ['parquet']
  provider                      30             82   ['parquet']
  encounter                     14             26   ['parquet']
  claim                         15             18   ['parquet']
  diagnosis                     56          3,824   ['parquet']
  medication                    56          1,909   ['parquet']
  procedure                     56          2,553   ['parquet']
  claim_line                    56          5,069   ['parquet']

Total entities: 9
Total rows written: 13,525
Total data files: 302


## Step 4 — Chaos Injection and Validation

**What we're about to do:** First, apply chaos injection to selected file drops —
simulating corrupted files, missing columns, and truncated data. Then run
Spindle's built-in validation gates to verify that quality checks catch the
injected problems.

**Why this matters:** Production pipelines break in subtle ways: a column
gets renamed upstream, a Parquet file is truncated mid-write, a schema
evolves without warning. Chaos injection lets you test your pipeline's
error handling, while validation gates confirm that your quality checks
actually catch these problems. Files with chaos applied should trigger
validation failures — if they don't, your quality gates have gaps.

**What to expect:** A chaos report showing affected files, followed by a
validation report showing which batches pass and which fail.

In [5]:
import pandas as pd
from sqllocks_spindle.chaos.config import ChaosConfig
from sqllocks_spindle.chaos.engine import ChaosEngine
import numpy as np

# Configure chaos using the actual ChaosEngine API
chaos_cfg = ChaosConfig(
    enabled=True,
    intensity="moderate",
    seed=42,
    warmup_days=0,
    chaos_start_day=0,
    categories={
        "value":       {"enabled": True, "weight": 0.30},
        "schema":      {"enabled": True, "weight": 0.20},
        "referential": {"enabled": False, "weight": 0.0},
        "temporal":    {"enabled": False, "weight": 0.0},
        "file":        {"enabled": False, "weight": 0.0},
        "volume":      {"enabled": False, "weight": 0.0},
    },
)
chaos = ChaosEngine(chaos_cfg)

# Read back some of the written Parquet files and apply chaos
affected_count = 0
affected_files = []

for file_path in drop_result.files_written[:10]:  # Process first 10 data files
    if not file_path.exists() or not str(file_path).endswith(".parquet"):
        continue
    try:
        df = pd.read_parquet(file_path)
        original_shape = df.shape

        corrupted = chaos.corrupt_dataframe(df.copy(), day=15)

        # Coerce int columns that got promoted to float back to nullable int
        for col in corrupted.columns:
            if col in df.columns:
                orig_dtype = df[col].dtype
                curr_dtype = corrupted[col].dtype
                if np.issubdtype(orig_dtype, np.integer) and np.issubdtype(curr_dtype, np.floating):
                    try:
                        corrupted[col] = corrupted[col].astype("Int64")
                    except (ValueError, TypeError):
                        pass

        corrupted = chaos.drift_schema(corrupted, day=15)

        if corrupted.shape != original_shape or not df.equals(corrupted):
            affected_count += 1
            affected_files.append(str(file_path.name))
            # Overwrite with corrupted version
            corrupted.to_parquet(file_path, index=False)
    except Exception as e:
        print(f"  Skipped {file_path.name}: {e}")

print("=== Chaos Injection Report ===")
print(f"  Files processed: {min(10, len(drop_result.files_written))}")
print(f"  Files affected:  {affected_count}")
print(f"  Files untouched: {min(10, len(drop_result.files_written)) - affected_count}")

if affected_files:
    print(f"\n=== Affected Files ===")
    for fname in affected_files:
        print(f"  {fname}")

# Validate: check that we can still read the non-corrupted files
print(f"\n=== Validation Check ===")
readable = 0
errors = 0
for file_path in drop_result.files_written:
    if not file_path.exists() or not str(file_path).endswith(".parquet"):
        continue
    try:
        df = pd.read_parquet(file_path)
        readable += 1
    except Exception as e:
        errors += 1
        print(f"  READ FAIL: {file_path.name} — {e}")

print(f"  Readable files: {readable}")
print(f"  Read errors:    {errors}")
print(f"\nSummary: {'ALL READABLE' if errors == 0 else f'{errors} FILES CORRUPTED'}")

  Skipped healthcare_facility_2024-01-01_00001.parquet: Cannot interpret '<StringDtype(na_value=nan)>' as a data type
  Skipped healthcare_facility_2024-01-04_00900.parquet: Cannot interpret '<StringDtype(na_value=nan)>' as a data type
  Skipped healthcare_facility_2024-01-02_00001.parquet: Cannot interpret '<StringDtype(na_value=nan)>' as a data type
  Skipped healthcare_facility_2024-01-03_00001.parquet: Cannot interpret '<StringDtype(na_value=nan)>' as a data type
  Skipped healthcare_facility_2024-01-04_00001.parquet: Cannot interpret '<StringDtype(na_value=nan)>' as a data type
  Skipped healthcare_facility_2024-01-06_00900.parquet: Cannot interpret '<StringDtype(na_value=nan)>' as a data type
  Skipped healthcare_facility_2024-01-05_00001.parquet: Cannot interpret '<StringDtype(na_value=nan)>' as a data type
  Skipped healthcare_facility_2024-01-06_00001.parquet: Cannot interpret '<StringDtype(na_value=nan)>' as a data type
  Skipped healthcare_facility_2024-01-10_00900.parquet: 

  Readable files: 302
  Read errors:    0

Summary: ALL READABLE


## Step 5 — Review the Complete Structure

**What we're about to do:** Display the full directory structure that was
generated — showing the date-partitioned folders, data files, manifests,
and done-flags.

**Why this matters:** This is the exact structure you would upload to your
Fabric Lakehouse `Files/` folder or ADLS landing zone. Your Data Pipeline
can be configured to watch this folder structure and process each daily
batch as it appears.

**What to expect:** A tree view of the output directory.

In [6]:
import os

base = config.base_path
print(f"=== File-Drop Directory Structure ===")
print(f"{base}/")

total_size = 0
for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    indent = "  " * (level + 1)
    folder = os.path.basename(root)
    if level > 0:
        print(f"{indent}{folder}/")
    sub_indent = "  " * (level + 2)
    for file in sorted(files):
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        total_size += size
        print(f"{sub_indent}{file:<35} {size:>10,} bytes")

print(f"\nTotal disk usage: {total_size:,.0f} bytes ({total_size/1024:.1f} KB)")
print(f"\nThis structure can be uploaded directly to a Fabric Lakehouse Files/ folder")
print(f"or an ADLS Gen2 landing zone for end-to-end pipeline testing.")

=== File-Drop Directory Structure ===
./f08_demo/
    healthcare/
      claim/
        dt=2024-01-01/
          _done                                       32 bytes
          _manifest.json                             313 bytes
          healthcare_claim_2024-01-01_00001.parquet      6,445 bytes
        dt=2024-01-02/
          _done                                       32 bytes
          _manifest.json                             313 bytes
          healthcare_claim_2024-01-02_00001.parquet      6,455 bytes
        dt=2024-01-03/
          _done                                       32 bytes
          _manifest.json                             313 bytes
          healthcare_claim_2024-01-03_00001.parquet      6,455 bytes
        dt=2024-01-05/
          _done                                       32 bytes
          _manifest.json                             313 bytes
          healthcare_claim_2024-01-05_00001.parquet      6,440 bytes
        dt=2024-01-06/
          _done           

## What's Next?

You've built a complete file-drop test harness — realistic daily drops with
late arrivals, chaos injection, and validation gates. This is everything you
need to test a Fabric ingestion pipeline end-to-end.

| Notebook | What You'll Learn |
|----------|-------------------|
| **F05: Chaos Pipeline** | Deep dive into chaos injection patterns |
| **F01: Medallion Architecture** | Build bronze/silver/gold layers from file drops |
| **T14: Day 2 — Incremental** | Generate CDC-style deltas for merge/upsert testing |
| **F09: Cross-Domain Enterprise** | Scale up to multi-domain enterprise datasets |
| **[F10 — Month-End Close](./F10_month_end_close.ipynb)** | Simulate 12 months of financial snapshots with quarter-end spikes and adjusting entries |

**Key takeaways:**
- `FileDropSimulator` creates date-partitioned file drops with manifests and done-flags
- Late arrival simulation tests your pipeline's handling of out-of-order data
- Chaos injection (corruption, missing columns, truncation) validates error handling
- Validation gates verify manifest accuracy and schema consistency
- The output structure maps directly to Fabric Lakehouse and ADLS landing zones